In [1]:
import sys, os, glob, torch
import torch.nn.functional as F

from pathlib import Path
from PIL import Image
from torchvision import transforms as T
from cellvit.models.cell_segmentation.cellvit_sam import CellViTSAM  # or CellViT / CellViT256 / CellViTUNI
from cellvit.models.cell_segmentation.cellvit_virchow import CellViTVirchow
from cellvit.utils.tools import unflatten_dict
from cellvit.inference.postprocessing_cupy import DetectionCellPostProcessorCupy
from cellvit.models.classifier.linear_classifier import LinearClassifier


USE_CUSTOM_CLS = False  # whether to use custom classifier for cell type classification

SAVE_ROOT = "/home/hqvo2/Projects/MICCAI_2026/CellViT-plus-plus/PathVG_results/CellViT-SAM-H-x40-AMP-001-1024x1024"

os.makedirs(SAVE_ROOT, exist_ok=True)

def apply_softmax_reorder(predictions: dict) -> dict:
    """Reorder and apply softmax on predictions

    Args:
        predictions(dict): Predictions

    Returns:
        dict: Predictions
    """
    predictions["nuclei_binary_map"] = F.softmax(
        predictions["nuclei_binary_map"], dim=1
    )
    predictions["nuclei_type_map"] = F.softmax(
        predictions["nuclei_type_map"], dim=1
    )
    predictions["nuclei_type_map"] = predictions["nuclei_type_map"].permute(
        0, 2, 3, 1
    )
    predictions["nuclei_binary_map"] = predictions["nuclei_binary_map"].permute(
        0, 2, 3, 1
    )
    predictions["hv_map"] = predictions["hv_map"].permute(0, 2, 3, 1)
    return predictions


# ensure local package is on path
repo_root = "/home/hqvo2/Projects/MICCAI_2026/CellViT-plus-plus"
sys.path.insert(0, repo_root)
os.environ["PYTHONPATH"] = repo_root + os.pathsep + os.environ.get("PYTHONPATH", "")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = "/home/hqvo2/Projects/MICCAI_2026/CellViT-plus-plus/checkpoints/SAM/CellViT-SAM-H-x40-AMP-001.pth"
# model_path = "/home/hqvo2/Projects/MICCAI_2026/CellViT-plus-plus/checkpoints/Virchow/CellViT-Virchow-x40-AMP-003.pth"
ckpt = torch.load(model_path, map_location="cpu")
run_conf = unflatten_dict(ckpt["config"], ".")

# Instantiate model with explicit args expected by CellViTSAM
model = CellViTSAM(
    model_path=None,  # weights come from the checkpoint below
    num_nuclei_classes=run_conf["data"]["num_nuclei_classes"],
    num_tissue_classes=run_conf["data"]["num_tissue_classes"],
    vit_structure=run_conf["model"]["backbone"],
    regression_loss=run_conf["model"].get("regression_loss", False),
).to(device).eval()
# model = CellViTVirchow(
#     model_virchow_path=None,  # weights come from the checkpoint below
#     num_nuclei_classes=run_conf["data"]["num_nuclei_classes"],
#     num_tissue_classes=run_conf["data"]["num_tissue_classes"]
# ).to(device).eval()

model.load_state_dict(ckpt["model_state_dict"])


# Custom classifiers
if USE_CUSTOM_CLS:
    classifier_path = "/home/hqvo2/Projects/MICCAI_2026/CellViT-plus-plus/checkpoints/classifier/sam-h/consep.pth"
    cls_model_checkpoint = torch.load(classifier_path, map_location="cpu")
    cls_run_conf = unflatten_dict(cls_model_checkpoint["config"], ".")

    cls_model = LinearClassifier(
        embed_dim=cls_model_checkpoint["model_state_dict"]["fc1.weight"].shape[1],
        hidden_dim=cls_run_conf["model"].get("hidden_dim", 100),
        num_classes=cls_run_conf["data"]["num_classes"],
        drop_rate=0,
    )

    cls_model.eval()
    label_map = cls_run_conf["data"]["label_map"]
    label_map = {int(k): v for k, v in label_map.items()}
    classifier = cls_model

    post = DetectionCellPostProcessorCupy(
        wsi=None, nr_types=run_conf['data']['num_nuclei_classes'], resolution=0.25,
        classifier=classifier
    )
else:
    post = DetectionCellPostProcessorCupy(
        wsi=None, nr_types=run_conf['data']['num_nuclei_classes'], resolution=0.25
    )

tfm = T.Compose([T.ToTensor()])  # add resize/normalize if needed

num_processed = 0
for img_path in glob.glob("/home/hqvo2/Projects/MICCAI_2026/datasets/RefPath/refpath_images/*.jpg"):
    save_path = Path(SAVE_ROOT) / (Path(img_path).stem + "_pred.pt")

    if os.path.exists(save_path):
        print(f"Skipping {img_path}, already processed.")
        continue

    print(img_path)
    img = Image.open(img_path)

    resized_img = img.resize((1024, 1024))

    print(resized_img.size)

    tensor = tfm(resized_img).unsqueeze(0).to(device)
    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16 if device.type == "cuda" else torch.bfloat16):
        pred = model.forward(tensor, retrieve_tokens=True)


    pred = apply_softmax_reorder(pred)
    post_pred = post.post_process_batch(pred)
    # Example: convert logits you need; the keys here are nuclei_binary_map, hv_map, nuclei_type_map
    # pred = {k: v.softmax(1) if k == "nuclei_type_map" else v for k, v in pred.items()}
    # post-processing/skipping coords is still up to you

    
    torch.save({'predictions': pred, 'post_processed': post_pred}, save_path)
    print(f"Saved predictions to {save_path}")

    # num_processed += 1
    # if num_processed >= 50:
    #     break


/project/hnguyen/hqvo2/miniconda3/envs/cellvit_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-04 21:34:06,445	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
/scratch/299444/ipykernel_102023/2234832898.py:54: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickl

Skipping /home/hqvo2/Projects/MICCAI_2026/datasets/RefPath/refpath_images/2100.jpg, already processed.
Skipping /home/hqvo2/Projects/MICCAI_2026/datasets/RefPath/refpath_images/21842.jpg, already processed.
Skipping /home/hqvo2/Projects/MICCAI_2026/datasets/RefPath/refpath_images/11759.jpg, already processed.
Skipping /home/hqvo2/Projects/MICCAI_2026/datasets/RefPath/refpath_images/19305.jpg, already processed.
Skipping /home/hqvo2/Projects/MICCAI_2026/datasets/RefPath/refpath_images/4571.jpg, already processed.
Skipping /home/hqvo2/Projects/MICCAI_2026/datasets/RefPath/refpath_images/16036.jpg, already processed.
Skipping /home/hqvo2/Projects/MICCAI_2026/datasets/RefPath/refpath_images/22393.jpg, already processed.
Skipping /home/hqvo2/Projects/MICCAI_2026/datasets/RefPath/refpath_images/7078.jpg, already processed.
Skipping /home/hqvo2/Projects/MICCAI_2026/datasets/RefPath/refpath_images/12250.jpg, already processed.
Skipping /home/hqvo2/Projects/MICCAI_2026/datasets/RefPath/refpath_

In [2]:
run_conf


{'gpu': 0,
 'data': {'dataset': 'PanNuke',
  'num_nuclei_classes': 6,
  'num_tissue_classes': 19},
 'model': {'backbone': 'SAM-H'},
 'training': {'drop_rate': 0,
  'attn_drop_rate': 0.1,
  'drop_path_rate': 0.1,
  'mixed_precision': True,
  'eval_every': 20},
 'transformations': {'randomsizedcrop': {'p': 0.1},
  'normalize': {'mean': [0.5, 0.5, 0.5], 'std': [0.5, 0.5, 0.5]}},
 'eval_checkpoint': 'latest_checkpoint.pth',
 'dataset_config': {'tissue_types': {'Adrenal_gland': 0,
   'Bile-duct': 1,
   'Bladder': 2,
   'Breast': 3,
   'Cervix': 4,
   'Colon': 5,
   'Esophagus': 6,
   'HeadNeck': 7,
   'Kidney': 8,
   'Liver': 9,
   'Lung': 10,
   'Ovarian': 11,
   'Pancreatic': 12,
   'Prostate': 13,
   'Skin': 14,
   'Stomach': 15,
   'Testis': 16,
   'Thyroid': 17,
   'Uterus': 18},
  'nuclei_types': {'Background': 0,
   'Neoplastic': 1,
   'Inflammatory': 2,
   'Connective': 3,
   'Dead': 4,
   'Epithelial': 5}}}

In [3]:
if USE_CUSTOM_CLS:
    cls_run_conf

In [4]:
pred.keys()

NameError: name 'pred' is not defined

In [ ]:
pred['nuclei_type_map']

tensor([[[[9.9902e-01, 4.1223e-04, 3.8981e-05, 2.9516e-04, 0.0000e+00,
           2.3544e-05],
          [1.0000e+00, 4.2140e-05, 5.5432e-06, 6.2883e-05, 0.0000e+00,
           1.3649e-05],
          [1.0000e+00, 2.1756e-05, 1.2517e-06, 3.5107e-05, 0.0000e+00,
           5.0664e-06],
          ...,
          [1.0000e+00, 2.6226e-06, 1.3113e-06, 1.0312e-05, 0.0000e+00,
           5.9605e-07],
          [1.0000e+00, 1.1146e-05, 7.0930e-06, 3.2902e-05, 0.0000e+00,
           1.9073e-06],
          [1.0000e+00, 8.0585e-05, 1.0014e-05, 9.6858e-05, 0.0000e+00,
           3.0398e-06]],

         [[1.0000e+00, 5.7697e-05, 9.6560e-06, 5.2631e-05, 0.0000e+00,
           1.4365e-05],
          [1.0000e+00, 8.5235e-06, 1.3709e-06, 1.3530e-05, 0.0000e+00,
           2.8610e-06],
          [1.0000e+00, 3.0398e-06, 3.5763e-07, 4.9472e-06, 0.0000e+00,
           7.7486e-07],
          ...,
          [1.0000e+00, 4.7684e-07, 1.7881e-07, 1.4901e-06, 0.0000e+00,
           1.1921e-07],
          [1.0000e

In [ ]:
import numpy as np
np.unique([el[1]['type'] for el in post_pred[1][0].items()])

array([0, 1, 2, 3, 5])

In [ ]:
post_pred[1][0]

{1: {'bbox': array([[  0, 363],
         [ 14, 385]]),
  'centroid': array([372.48192771,   5.51004016]),
  'contour': array([[364,   0],
         [363,   1],
         [363,   9],
         [364,  10],
         [364,  11],
         [365,  12],
         [366,  12],
         [367,  13],
         [371,  13],
         [372,  12],
         [374,  12],
         [376,  10],
         [378,  10],
         [379,   9],
         [380,   9],
         [383,   6],
         [383,   5],
         [384,   4],
         [384,   1],
         [383,   0]], dtype=int32),
  'type_prob': 0.9999999959839357,
  'type': 0},
 2: {'bbox': array([[  0, 556],
         [ 20, 585]]),
  'centroid': array([569.93807339,   8.29357798]),
  'contour': array([[557,   0],
         [556,   1],
         [556,   6],
         [557,   7],
         [557,   8],
         [558,   9],
         [558,  10],
         [560,  12],
         [560,  13],
         [561,  14],
         [562,  14],
         [564,  16],
         [565,  16],
         

In [ ]:
post_pred

(tensor([[[  0.,   0.,   0.,  ...,   0.,   0.,   0.],
          [  0.,   0.,   0.,  ...,   0.,   0.,   0.],
          [  0.,   0.,   0.,  ...,   0.,   0.,   0.],
          ...,
          [  0.,   0.,   0.,  ..., 107., 107., 107.],
          [  0.,   0.,   0.,  ..., 107., 107., 107.],
          [  0.,   0.,   0.,  ..., 107., 107., 107.]]]),
 [{1: {'bbox': array([[  0, 363],
           [ 14, 385]]),
    'centroid': array([372.48192771,   5.51004016]),
    'contour': array([[364,   0],
           [363,   1],
           [363,   9],
           [364,  10],
           [364,  11],
           [365,  12],
           [366,  12],
           [367,  13],
           [371,  13],
           [372,  12],
           [374,  12],
           [376,  10],
           [378,  10],
           [379,   9],
           [380,   9],
           [383,   6],
           [383,   5],
           [384,   4],
           [384,   1],
           [383,   0]], dtype=int32),
    'type_prob': 0.9999999959839357,
    'type': 0},
   2: {